# 案例：中心控制器编排与领域智能体协作

> Last updated: 2026-11-18

## 核心学习目标

本案例以多城市旅行规划为对象，学习如何设计领域智能体角色，并让中心编排器根据任务能力完成角色匹配、依赖排序和统一调度。

## 案例设计

## 中心编排流程

1. 中心智能体先判断请求类型。非旅行请求直接由大模型回复，不启动后续复杂流程。
2. 旅行请求先复用事实调查框架识别已知条件、待查信息和不确定项，再将目标拆解为子任务并分析依赖。
3. 独立的路由模块根据领域智能体职责、工具能力和输入输出定义，为每个子任务匹配最合适的智能体角色。
4. 中心控制器按依赖顺序调用领域智能体，全部任务完成后由最终规划智能体生成统一答复。

`fact_survey.py` 提供跨章节事实调查模板；`task_plan.py` 负责任务拆解和依赖校验；`agent_routing.py` 独立负责任务到智能体的匹配；`domain_agents.py` 定义领域角色和能力；`orchestrator.py` 用代码控制请求分诊、规划、路由和执行。

**学习重点**：
1. **领域智能体角色设定**：理解如何用职责描述、工具、输入输出和提示词定义一个领域智能体。
2. **任务与智能体角色匹配**：理解拆解后的子任务如何根据能力边界匹配到合适的领域角色。
3. **中心化编排**：理解中心编排器如何完成事实调查、任务拆解、依赖排序、执行调度和最终结果选择。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 模型访问 | 模型创建、文本调用、JSON 与结构化输出 | `llm.py` |
| 请求分诊 | 区分旅行任务与非旅行任务，非旅行任务直接回复 | `orchestrator.py` |
| 事实调查 | 四类事实分析与相对日期换算 | `fact_survey.py` |
| 任务规划 | 任务拆解、依赖分析和执行顺序校验 | `task_plan.py` |
| 角色匹配 | 子任务到领域智能体的能力路由 | `agent_routing.py` |
| 中心编排 | 统一执行、分层调度、最终回复选择与记忆写入 | `orchestrator.py` |
| 领域角色 | 领域智能体的职责、工具、输入输出和提示词 | `domain_agents.py` |
| 长期记忆 | 复用 Chapter-3 的短期上下文、Chroma 检索和记忆写入 | `memory_adapter.py` |
| 最终规划 | 汇总领域结果并选择最终用户答复 | `orchestrator.py` |

## 1. 环境与模块导入

本案例使用仓库根目录的共享 `.venv`。环境变量统一从仓库根目录 `.env` 加载。

In [1]:
import sys
from pathlib import Path

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

chapter6_dir = Path.cwd()
if not (chapter6_dir / "config.py").exists():
    chapter6_dir = Path.cwd() / "Chapter-6"
if not (chapter6_dir / "config.py").exists():
    raise FileNotFoundError("Could not locate Chapter-6/config.py")
if str(chapter6_dir) not in sys.path:
    sys.path.insert(0, str(chapter6_dir))

from config import BOOK_ROOT, CH6_DIR, CHROMA_DIR, load_project_dotenv

env_path = load_project_dotenv()
print("Chapter-6:", CH6_DIR)
print("项目根目录:", BOOK_ROOT)
print("环境文件:", env_path)
print("Chroma 目录:", CHROMA_DIR)

Chapter-6: E:\Book\agent-systems-book\Chapter-6
项目根目录: E:\Book\agent-systems-book
环境文件: E:\Book\agent-systems-book\.env
Chroma 目录: E:\Book\agent-systems-book\Chapter-6\chroma_memory


In [2]:
import importlib
import io
import json
import uuid
from contextlib import redirect_stdout

import agent_routing
import domain_agents
import fact_survey
import memory_adapter
import orchestrator
import task_plan

for module in (
    fact_survey,
    task_plan,
    agent_routing,
    domain_agents,
    memory_adapter,
    orchestrator,
):
    importlib.reload(module)

from domain_agents import DOMAIN_AGENT_SPECS
from orchestrator import CENTRAL_AGENT_SYSTEM_PROMPT, CentralOrchestrator

print("模块导入完成")

模块导入完成


## 2. 领域智能体角色设定

本章定义了 **6 个领域智能体**，分别承担旅行规划中的不同专业职责：

| 领域智能体 | 主要职责 |
|---|---|
| `WeatherAgent` | 根据城市和确切日期查询天气预报；每次请求只查询一个城市、一个日期的天气 |
| `AttractionAgent` | 根据城市和旅行偏好推荐景点；每次请求只检索一个城市的景点候选 |
| `HotelAgent` | 根据城市推荐酒店和住宿；每次请求只执行一次酒店推荐操作 |
| `RestaurantAgent` | 根据位置推荐餐厅；每次请求只查询一组餐厅推荐结果 |
| `FlightAgent` | 根据出发城市、到达城市和确切日期查询航班信息；每次请求只查询一组航线结果 |
| `ItineraryAgent` | 综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复 |

这些角色的详细定义集中在 `domain_agents.py`：`AgentSpec` 声明角色身份、职责描述、工具、输入输出和系统提示词，`ToolSpec` 声明可调用工具及其参数，`DomainAgentRegistry` 将定义汇总为后续任务规划和智能体路由使用的能力目录。

In [3]:
print("=== 领域智能体角色定义 ===\n")
for spec in DOMAIN_AGENT_SPECS.values():
    print(f"[{spec.name}] {spec.description}")
    for tool_spec in spec.tools:
        print(f"  工具: {tool_spec.name}")
        print(f"  输入: {tool_spec.parameters}")
        print(f"  输出: {tool_spec.returns}")
    if not spec.tools:
        print("  工具: 无（只负责最终规划与总结）")
        print(f"  输入: {spec.input_fields}")
        print(f"  输出: {spec.output_fields}")
    prompt_preview = " ".join(spec.system_prompt.split())
    print(f"  提示词: {prompt_preview[:140]}...")
    print()

=== 领域智能体角色定义 ===

[WeatherAgent] 根据城市和确切日期查询天气预报；每次请求只查询一个城市、一个日期的天气。
  工具: get_weather
  输入: ['city', 'date']
  输出: ['city', 'date', 'forecast', 'count', 'data_source', 'note']
  提示词: 你是天气查询领域智能体。 当前日期（本地时间）：2026-11-18 职责： 1. 使用可用天气工具完成当前天气子任务 2. 工具结果为空时，明确说明第 6 章尚未接入实时天气数据 3. 多日查询应基于工具实际返回的数据回答，不得补造缺失日期 4. 当前消息是明确天气子任务时，专...

[AttractionAgent] 根据城市和旅行偏好推荐景点；每次请求只检索一个城市的景点候选。
  工具: recommend_attractions
  输入: ['city', 'preferences', 'limit']
  输出: ['city', 'attractions', 'count', 'data_source']
  提示词: 你是景点推荐领域智能体。 职责： 1. 使用可用景点工具完成当前子任务 2. 只能从工具返回的景点候选中推荐，不得补充未返回的景点 3. 推荐理由只能基于名称、地址、区域、评分和类型等已有字段，不得编造简介、开放时间或门票 4. 非景点相关问题，回复：我只能协助景点推荐 注意： ...

[HotelAgent] 根据城市推荐酒店和住宿；每次请求只执行一次酒店推荐操作。
  工具: recommend_hotel
  输入: ['city', 'preferences', 'budget_cny_per_night_max']
  输出: ['city', 'hotels', 'count', 'data_source']
  提示词: 你是酒店推荐领域智能体。 规则： 1. 使用可用酒店工具完成当前子任务。 2. 安静、舒适等主观属性不由工具提供，不得声称已经核验。 3. 工具返回酒店候选后，只能基于区域、地址、参考价格、评分和类型推荐 3–5 家酒店。 每家需包含：名称、地址/位置、参考价格、评分（如有）、1...

[RestaurantAgent] 根据位置推

## 3. 用户输入

本案例使用一个多城市旅行规划问题。该请求同时包含出发地、目的地、时间、交通、住宿、天气和美食需求，适合作为后续任务拆解和依赖分析的输入。

In [4]:
user_query = """
你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序……
我想从北京出发，大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、
天气情况和美食攻略。我喜欢住安静的酒店，预算每晚不超过800元。
"""

print("用户输入：")
print(user_query.strip())

用户输入：
你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序……
我想从北京出发，大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、
天气情况和美食攻略。我喜欢住安静的酒店，预算每晚不超过800元。


## 4. 中心智能体 System Prompt 与请求分诊
中心 System Prompt 定义在 `orchestrator.py`，负责把请求分为两类：非旅行请求直接回答；旅行请求进入后续规划、路由和执行流程。

下方调用 `decide_request_route()` 执行请求分诊，并确认当前用户输入进入旅行任务链路。

In [5]:
from IPython.display import Markdown, display

display(Markdown("```text\n" + CENTRAL_AGENT_SYSTEM_PROMPT + "\n```"))

ORCHESTRATION_THREAD_ID = "central_orchestration_entry_001"
orchestrator_instance = CentralOrchestrator(enable_memory=True)
request_route = await orchestrator_instance.decide_request_route(user_query)

assert request_route["route"] == "travel"
print(f"请求类型: {request_route['route']}")

```text
你是旅行规划应用的中心编排智能体，负责请求分诊和后续流程控制。

## 任务类型判断
将用户请求分为两类：

1. 旅行任务
包括旅行行程规划，以及天气、景点、酒店、餐厅、航班、交通等旅行相关信息查询。这类任务需要后续的事实调查、任务规划、领域智能体分配和执行，不能直接凭模型知识作答。

2. 非旅行任务
包括问候、写作、编程、常识问答、数学计算以及与旅行无关的信息查询。这类任务不需要启动任务规划或多智能体执行。

## 处理规则
1. 如果是非旅行任务：直接使用你自己的通用能力回答；


2. 如果是旅行任务：
   - 不要直接生成最终答案；
   - 将任务交给后续流程：先完成事实调查、任务拆解和依赖分析，再匹配领域智能体，最后按执行计划调用领域智能体；
   - response 字段留空，等待后续流程生成最终答案。

## 输出格式
只输出 JSON 对象，不要输出解释文字：

非旅行任务：
{"route": "direct", "response": "直接给用户的完整回答"}

旅行任务：
{"route": "travel", "response": ""}

```

✓ 长期记忆系统已启用（Chapter-3）


请求类型: travel


## 5. 事实调查与长期记忆检索

先调用 `task_plan.survey_facts()`，区分用户已给出的事实、待查信息、待推导信息和不确定项；随后通过 `MemoryAdapter` 复用 Chapter 3 的长期记忆检索能力，查找与当前请求相关的用户偏好。

事实调查结果和长期记忆会作为后续任务拆解的背景信息，不直接作为领域事实答案。

In [6]:
pre_survey = await task_plan.survey_facts(
    orchestrator_instance.llm,
    user_query,
)
print("=== 事实调查结果 ===\n")
print(json.dumps(
    {key: value for key, value in pre_survey.items() if key != "raw_text"},
    ensure_ascii=False,
    indent=2,
))

memories = []
if orchestrator_instance.memory_adapter:
    hits = await orchestrator_instance.memory_adapter.search_memories_async(user_query)
    memories = orchestrator_instance.memory_adapter.format_memories_for_plan(hits)
print(f"\n检索到的长期记忆: {len(memories)} 条")

Building prefix dict from the default dictionary ...


Loading model from cache C:\Users\liuhui36\AppData\Local\Temp\jieba.cache


=== 事实调查结果 ===

{
  "given_facts": [
    "出发地：北京",
    "目的地城市：上海、苏州、杭州",
    "旅行时间：五一假期（通常指5月1日至5月5日）",
    "住宿偏好：安静的酒店",
    "住宿预算：每晚不超过800元",
    "需求内容：行程路线、酒店推荐、天气情况、美食攻略"
  ],
  "facts_to_lookup": [
    "2027年五一劳动节的具体放假安排及日期范围",
    "2027年5月初上海、苏州、杭州的历史平均天气数据或长期天气预报",
    "上海、苏州、杭州符合“安静”且“每晚800元以下”条件的具体酒店列表及实时价格",
    "北京至上海、以及上海-苏州-杭州之间的高铁或交通时刻表与票价",
    "上海、苏州、杭州当地的特色美食推荐及知名餐厅",
    "各城市主要景点的开放时间、门票价格及预约政策"
  ],
  "facts_to_derive": [
    "基于地理位置和交通便捷性的最优城市游览顺序（如：北京->上海->苏州->杭州 或 北京->杭州->苏州->上海）",
    "每日具体的行程安排（包括景点游览时间、交通耗时、餐饮时间）",
    "总旅行预算估算（含交通、住宿、餐饮、门票）",
    "根据天气情况调整的室内/室外活动建议"
  ],
  "educated_guesses": [
    "用户可能倾向于高铁出行，因为京沪及长三角城际高铁网络发达且便捷",
    "“安静”的酒店可能指位于非核心商业区、巷弄深处或主打园林/度假风格的精品酒店或民宿",
    "五一期间为旅游高峰期，酒店价格可能上浮，800元预算在核心区域选择有限，可能需要提前预订或选择稍偏地段",
    "经典的江南游览路线通常为“上海进，杭州出”或反之，苏州位于中间，适合串联"
  ]
}


Loading model cost 1.207 seconds.


Prefix dict has been built successfully.



检索到的长期记忆: 1 条


## 6. 任务拆解与依赖分析

任务拆解、依赖识别和拓扑排序已在 Chapter 4 的任务分解与规划案例中详细学习，本章不再分别展开这两部分。

这里把任务拆解和依赖分析合并为一个代码单元格，直接调用 `task_plan.create_task_plan()`，一次得到业务子任务、任务依赖和执行顺序。

In [7]:
task_plan_result = await task_plan.create_task_plan(
    orchestrator_instance.llm,
    user_query,
    pre_survey,
    memories,
    orchestrator_instance.agent_registry,
)

print("=== 任务拆解与依赖分析结果 ===\n")
for task in task_plan_result["tasks"]:
    print(f"[{task['task_id']}] {task['description']}")
    print(f"  依赖: {task.get('depends_on', [])}")

print("\n执行顺序:", " → ".join(task_plan_result["execution_order"]))

=== 任务拆解与依赖分析结果 ===

[T1] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月1日的天气
  依赖: []
[T2] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月2日的天气
  依赖: []
[T3] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月3日的天气
  依赖: []
[T4] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月4日的天气
  依赖: []
[T5] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月5日的天气
  依赖: []
[T6] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月1日的天气
  依赖: []
[T7] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月2日的天气
  依赖: []
[T8] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月3日的天气
  依赖: []
[T9] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月4日的天气
  依赖: []
[T10] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月5日的天气
  依赖: []
[T11] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月1日的天气
  依赖: []
[T12] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月2日的天气
  依赖: []
[T13] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月3日的天气
  依赖: []
[T14] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月4日的天气
  依赖: []
[T15] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月5日的天气
  依赖: []
[T16] 根据城市和旅行偏好推荐景点：检索城市上海的景点候选
  依赖: []
[T17] 根据城市和旅行偏好推荐景点：检索城市苏州的景点候选
  依赖: []
[T18] 根据城市和旅行偏好推荐景点：检索城市杭州的景点候选
  依赖: []
[T19] 根据城市推荐酒店和住宿：推荐城市上海符合安静偏好且每晚不超过800元的酒店
  依赖: []
[T20] 根据城市推荐酒店和住宿：推荐城市苏州符合安静偏好且每晚不超过800元的酒店
  依

## 7. 智能体路由提示词

任务规划完成后，`agent_routing.py` 负责为每个业务子任务选择领域智能体，并从事实调查结果中提取该智能体工具需要的参数。

路由提示词接收四类输入：可用领域智能体及技能、事实调查结果、待分配任务、已验证的依赖关系。模型内部输出结构化路由记录，代码随后校验智能体名称、过滤无效参数，并保留任务原文和依赖关系。

In [8]:
from agent_routing import AGENT_ROUTING_PROMPT

display(Markdown("### 智能体路由提示词\n\n```text\n" + AGENT_ROUTING_PROMPT + "\n```"))

### 智能体路由提示词

```text
你是中心智能体的「子任务路由模块」。根据已拆解的子任务，为每个子任务选择最合适的领域智能体并提取调用参数。

## 可用领域智能体
{agent_team}

## 事实调查结果
{pre_survey_json}

## 子任务列表
{subtasks_json}

## 依赖分析结果
以下 depends_on 已通过本地校验，必须原样使用：
{depends_on_json}

## 输出要求
只输出 JSON 数组，其中每个元素对应一个子任务：
[
  {{
    "task_id": "T1",
    "agent": "WeatherAgent",
    "params": {{"city": "上海", "date": "2026-05-01"}}
  }}
]

规则：
1. agent 必须是可用领域智能体之一。
2. 每个子任务只分配一个 agent。
3. params 字段名必须匹配该 agent 的 inputSchema。
4. 必须为每个 task_id 输出一条记录。
5. 不要输出 description、depends_on 或其他解释文字，任务原文和依赖关系由本地代码保留。

```

## 8. 智能体路由执行与结果

调用独立的 `agent_routing.route_to_agents()`，把每个业务子任务分配给最合适的领域智能体。路由结果保留任务原文、验证后的依赖关系和提取出的工具参数；智能体名称和参数字段都会经过本地校验。

下发给领域智能体的是自然语言子任务，而不是路由模块内部的 JSON。下面展示本章最关心的问题：“子任务 → 领域智能体”的匹配结果。

In [9]:
depends_map = {
    task["task_id"]: list(task.get("depends_on") or [])
    for task in task_plan_result["tasks"]
}
validated_order = task_plan_result["execution_order"]
assigned_tasks = await orchestrator_instance.agent_routing.route_to_agents(
    task_plan_result["tasks"],
    validated_order,
    depends_map,
    pre_survey,
)
execution_plan = {
    "pre_survey": task_plan_result["pre_survey"],
    "pre_survey_raw": task_plan_result["pre_survey_raw"],
    "retrieved_memories": task_plan_result["retrieved_memories"],
    "total_goal": task_plan_result["total_goal"],
    "plan_fallback": task_plan_result["plan_fallback"],
    "subtasks": assigned_tasks,
    "execution_order": validated_order,
}

print("=== 智能体路由结果 ===\n")
for task in assigned_tasks:
    print(f"[{task['task_id']}] {task['description']}")
    print(f"  → {task.get('agent') or '未分配'}")
    print(f"  依赖: {task.get('depends_on', [])}")
    if task.get("assignment_error"):
        print(f"  路由错误: {task['assignment_error']}")
    print()

=== 智能体路由结果 ===

[T1] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月1日的天气
  → WeatherAgent
  依赖: []

[T2] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月2日的天气
  → WeatherAgent
  依赖: []

[T3] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月3日的天气
  → WeatherAgent
  依赖: []

[T4] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月4日的天气
  → WeatherAgent
  依赖: []

[T5] 根据城市和确切日期查询天气预报：查询城市上海在2027年5月5日的天气
  → WeatherAgent
  依赖: []

[T6] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月1日的天气
  → WeatherAgent
  依赖: []

[T7] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月2日的天气
  → WeatherAgent
  依赖: []

[T8] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月3日的天气
  → WeatherAgent
  依赖: []

[T9] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月4日的天气
  → WeatherAgent
  依赖: []

[T10] 根据城市和确切日期查询天气预报：查询城市苏州在2027年5月5日的天气
  → WeatherAgent
  依赖: []

[T11] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月1日的天气
  → WeatherAgent
  依赖: []

[T12] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月2日的天气
  → WeatherAgent
  依赖: []

[T13] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月3日的天气
  → WeatherAgent
  依赖: []

[T14] 根据城市和确切日期查询天气预报：查询城市杭州在2027年5月4日的天气
  → WeatherAgent
  依赖: []

[T15] 根据城市和确切日期查询天气预报：查询城市

## 9. 领域智能体执行结果

调用 `orchestrator.execute_plan()`，按依赖层顺序执行领域智能体；同一层中，仅声明支持并发的智能体并行执行。

存在依赖时，当前领域智能体会收到前序子任务、前序任务下发的领域智能体以及该智能体的回复。工具原始结果不继续向下游传递。`message_records` 用于保存并展示这些实际下发内容。

In [10]:
run_tag = uuid.uuid4().hex[:8]
agent_messages = []
execution_log = io.StringIO()
with redirect_stdout(execution_log):
    subtask_results = await orchestrator_instance.execute_plan(
        execution_plan,
        ORCHESTRATION_THREAD_ID,
        run_tag,
        message_records=agent_messages,
    )

print("=== 带依赖关系的领域智能体实际输入 ===\n")
for record in agent_messages:
    message = record["message"]
    if "前序依赖结果" not in message:
        continue
    print(f"[{record['task_id']}] → {record['agent']}")
    print(message)
    print()


print("=== 领域智能体执行结果 ===\n")
for task_id in execution_plan["execution_order"]:
    task_result = subtask_results.get(task_id, {})
    tool_data = task_result.get("tool_data")
    fields = list(tool_data) if isinstance(tool_data, dict) else []
    summary = " ".join((task_result.get("agent_summary") or "").split())
    print(f"[{task_id}] {task_result.get('agent', '?')}")
    print(f"  智能体输出: {summary[:240]}{'…' if len(summary) > 240 else ''}")
    print()

=== 带依赖关系的领域智能体实际输入 ===

[T26] → ItineraryAgent
【子任务 T26 · ItineraryAgent】
请完成下方子任务，并在调用工具后用中文总结结果。
即使任务说明很长或提到多城市/其他模块，也只完成本子任务；不要回复「我只能协助…」拒答。
子任务：综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复
工具参数：cities=['上海', '苏州', '杭州']，dates=['2027-05-01', '2027-05-02', '2027-05-03', '2027-05-04', '2027-05-05']，preferences=安静的酒店，美食攻略，departure_city=北京，destination_city=上海，days=5
前序依赖结果（作为本任务输入）：
- 前序任务 T1：根据城市和确切日期查询天气预报：查询城市上海在2027年5月1日的天气
  下发给智能体：WeatherAgent
  智能体回复：查询结果：第 6 章尚未接入实时天气数据，因此无法提供上海在 2027-05-01 的具体天气预报。后续章节会补充相关数据源。
- 前序任务 T2：根据城市和确切日期查询天气预报：查询城市上海在2027年5月2日的天气
  下发给智能体：WeatherAgent
  智能体回复：查询结果：第 6 章尚未接入实时天气数据，因此无法提供上海在 2027-05-02 的具体天气预报。后续章节会补充相关数据源。
- 前序任务 T3：根据城市和确切日期查询天气预报：查询城市上海在2027年5月3日的天气
  下发给智能体：WeatherAgent
  智能体回复：查询结果：第 6 章尚未接入实时天气数据，因此无法提供上海在 2027-05-03 的具体天气预报。后续章节会补充相关数据源。
- 前序任务 T4：根据城市和确切日期查询天气预报：查询城市上海在2027年5月4日的天气
  下发给智能体：WeatherAgent
  智能体回复：查询结果：第 6 章尚未接入实时天气数据，因此无法提供上海在 2027-05-04 的具体天气预报。后续章节会补充相关数据源。
- 前序任务 T5：根据城市和确切日期查询天气预报：查询城市上海在2027年5月5日的天气
  下发给智能

## 10. 最终旅行规划

调用 `compose_final_response()`，选择最后生成的旅行规划结果作为最终答复。`ItineraryAgent` 不调用领域工具，只综合依赖任务返回的天气、景点、酒店、餐厅、航班和用户偏好生成日程。

In [11]:
final_log = io.StringIO()
with redirect_stdout(final_log):
    final_response = orchestrator_instance.compose_final_response(
        subtask_results,
        execution_plan["execution_order"],
    )

if orchestrator_instance.memory_adapter:
    orchestrator_instance.memory_adapter.record_turn(
        ORCHESTRATION_THREAD_ID,
        user_query,
        final_response,
    )
    await orchestrator_instance.memory_adapter.ingest(
        f"用户请求: {user_query.strip()}\n偏好摘要: {final_response[:500]}",
        memory_type="preference",
    )

display(Markdown("### 最终旅行规划\n\n" + final_response))

### 最终旅行规划

很抱歉，基于当前的依赖查询结果，**无法为您生成完整的旅行日程**。

主要原因如下：
1.  **关键数据缺失**：
    *   **航班**：无法获取从北京到上海的航班信息（2027-05-01）。
    *   **景点**：上海和苏州未检索到有效景点；仅杭州有景点候选，但缺乏其他城市的支持。
    *   **酒店**：上海、苏州、杭州三地均未能返回符合“安静”偏好及预算要求的酒店列表（工具报错或无数据）。
    *   **餐厅**：上海、苏州、杭州三地均无餐厅推荐数据返回。
    *   **天气**：所有涉及日期和城市的天气预报数据均为空。

由于缺乏交通、住宿、餐饮和大部分景点的基础数据，无法构建可行的每日行程。建议您检查数据源连接状态，或尝试放宽搜索条件（如移除主观偏好“安静”、提高酒店预算）后重新发起查询。

## 11. 非旅行请求直答

中心智能体识别出非旅行任务后，直接调用大模型回答，不进入事实调查、任务拆解、依赖分析和多智能体执行。

In [12]:
direct_query = "请用一句话解释什么是二分查找。"
with redirect_stdout(io.StringIO()):
    direct_result = await orchestrator_instance.process_request(
        direct_query,
        thread_id="direct_query_entry_001",
    )

assert direct_result["route"] == "direct"
assert direct_result["execution_plan"] is None
print(direct_result["final_response"])

二分查找是一种在有序数组中查找特定元素的算法，通过反复将搜索区间减半来快速定位目标值。


## 12. 小结

- 领域智能体通过职责描述、工具、输入输出和提示词统一定义。
- 中心智能体负责请求分诊，后续规划、路由和执行顺序由代码控制。
- `task_plan.py` 复用 Chapter 2 和 Chapter 4 的能力，负责事实调查、任务拆解和依赖分析。
- `agent_routing.py` 独立完成任务到领域智能体的匹配，路由结构不直接暴露给领域智能体。
- 存在依赖时，领域智能体接收前序任务、前序智能体和前序回复，不接收工具原始结果。
- `ItineraryAgent` 综合领域执行结果，生成完整旅行答复。